##### Note - Create a worksheet for this raw code cause we are using snowpark framework of snowflake So create a worksheet with .py extension and paste the code and run it. 

In [ ]:
#########################################################################
# Step 1 - Import Snowpark Packages
#########################################################################

import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, sum, year, quarter, to_date, concat, lit


#########################################################################
# Step 2 - Setup the current database and schema
#########################################################################

def main(session: snowpark.Session):

    # Set schema
    session.sql("USE SCHEMA SNOWPARK_DB.TRANSFORMED").collect()

    #####################################################################
    # Curation 1 - Only Delivered Orders
    #####################################################################

    df_global_sales_order_delivered = (
        session
        .table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER")
        .filter(col("SHIPPING_STATUS") == "Delivered")
    )

    df_global_sales_order_delivered.write.mode("overwrite").save_as_table(
        "SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_DELIVERED"
    )


    #####################################################################
    # Curation 2 - Total Sales by Mobile Brand & Model
    #####################################################################

    df_global_sales_order_brand = (
        session
        .table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER")
        .withColumn("TOTAL_PRICE_NUM", col("TOTAL_PRICE").cast("float"))
        .groupBy(
            col("MOBILE_BRAND"),
            col("MOBILE_MODEL")
        )
        .agg(
            sum(col("TOTAL_PRICE_NUM")).alias("TOTAL_SALES_AMOUNT")
        )
    )

    df_global_sales_order_brand.write.mode("overwrite").save_as_table(
        "SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_BRAND"
    )


    #####################################################################
    # Curation 3 - Sales by Country / Year / Quarter
    #####################################################################

    df_global_sales_order_country = (
        session
        .table("SNOWPARK_DB.TRANSFORMED.GLOBAL_SALES_ORDER")
        .withColumn("TOTAL_PRICE_NUM", col("TOTAL_PRICE").cast("float"))
        .withColumn("QUANTITY_NUM", col("QUANTITY").cast("int"))
        .groupBy(
            col("COUNTRY"),
            year(to_date(col("ORDER_DATE"))).alias("YEAR"),
            concat(
                lit("Q"),
                quarter(to_date(col("ORDER_DATE")))
            ).alias("QUARTER")
        )
        .agg(
            sum(col("QUANTITY_NUM")).alias("TOTAL_SALES_VOLUME"),
            sum(col("TOTAL_PRICE_NUM")).alias("TOTAL_SALES_AMOUNT")
        )
    )

    df_global_sales_order_country.write.mode("overwrite").save_as_table(
        "SNOWPARK_DB.CURATED.GLOBAL_SALES_ORDER_COUNTRY"
    )

    return "Code to load the curated schema table ran successfully"

## Output

The curated layer contains analytics-ready datasets for reporting and business analysis.